# 02 · Similarity by build year

**Purpose.** Decide how to score the **age** block: how similar a marketed listing's build year is to the
portfolio's, for the brief's prompt 2 (a homogeneity score for each dimension, age among them) and as one
input to prompt 1 (the most similar properties).

The data is the cleaned output of `indurent_matching.data` (see `01_eda.ipynb` for the cleaning decisions).
The notebook ends with decisions to carry into `src/indurent_matching/similarity.py`.

## 1. The question

**Age homogeneity** asks: *is this listing's build year typical of the portfolio?* A listing built in a year
when Indurent owns plenty of stock is homogeneous with it on age; one built in a year it hardly owns is not.

We start with build year because it's the most straightforward: one feature, recorded for almost every unit.
The method below is chosen for the shape of the build-year data, not assumed in advance.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

from indurent_matching.data import load_marketed, load_portfolio

# Store each chart's data in the notebook itself; JupyterLab draws it with the Plotly extension, and the HTML
# copy is a fallback for other viewers.
pio.renderers.default = "plotly_mimetype+notebook_connected"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

# Portfolio build years, leaving out any flagged doubtful (none are at present).
years = portfolio.loc[~portfolio["build_year_doubtful"], "build_year"].dropna()
print(f"Portfolio: {portfolio['build_year'].notna().sum():,} of {len(portfolio):,} units have a build year; "
      f"leaving out {portfolio['build_year_doubtful'].sum()} flagged doubtful gives {len(years):,}")
marketed[["industrial_estate_name", "unit_name", "build_year"]]

Portfolio: 1,234 of 1,250 units have a build year; leaving out 0 flagged doubtful gives 1,234


,industrial_estate_name,unit_name,build_year
listing_id,,,
M1,Cherry Lane,Unit 10,1977.0
M2,Tech Hub,Unit 8,NaN
M3,Spitfire Park,Unit 42,1976.0
M4,Stable Lane,Unit 14,2000.0
M5,Chancery Depot,Unit 2,1966.0


* **No build years are flagged doubtful.** Granite Mews 59's (1939-1953) were suspected at first (EDA 2.9), but
  a 1940s industrial building is plausible, and the EPC A ratings that seemed to contradict them can't be relied
  on. Only its eaves are flagged. Any build year flagged in future would be left out here.
* Four of the five listings have a build year. **M2 (Tech Hub) has none**, so its age score will be "not
  available", not a low score.

## 2. The distribution

In [2]:
fig = px.histogram(years, x="build_year", nbins=int(years.max() - years.min() + 1),
                   title="Portfolio build years (dashed lines: marketed listings' build years)",
                   labels={"build_year": "Build year"}, height=420)
fig.update_layout(showlegend=False, yaxis_title="Units")
for year in marketed["build_year"].dropna():
    fig.add_vline(x=year, line_dash="dash", line_color="firebrick")
fig.show()

* From **1970 to 2021 the distribution is roughly flat**: no decade dominates, and no year stands out apart from
  ordinary year-to-year noise.
* Before 1970 there's a **thin tail** back to 1939.
* **2022 is a spike**, the tallest bar by far and the last year in the data (section 3 looks at it).
* The listings: **M1 (1977), M3 (1976) and M4 (2000)** sit in the flat part, among plenty of portfolio stock.
  **M5 (1966)** is in the thin tail.

In [3]:
per_year = years.value_counts().sort_index()
periods = pd.cut(years, [1900, 1970, 1980, 1990, 2000, 2010, 2020, 2022, 2023], right=False,
                 labels=["before 1970", "1970s", "1980s", "1990s", "2000s", "2010s", "2020-21", "2022"])
summary = years.groupby(periods, observed=False).size().to_frame("units")
summary["years covered"] = [int(1970 - years.min()), 10, 10, 10, 10, 10, 2, 1]
summary["units per year"] = (summary["units"] / summary["years covered"]).round(1)
summary

,units,years covered,units per year
build_year,,,
before 1970,46,31,1.5
1970s,202,10,20.2
1980s,217,10,21.7
1990s,251,10,25.1
2000s,232,10,23.2
2010s,175,10,17.5
2020-21,31,2,15.5
2022,80,1,80.0


* **18-25 units a year** in every decade from the 1970s to the 2010s: close to uniform, as the histogram
  suggests. The 2010s and 2020-21 are slightly lower, which fits newer stock taking time to be acquired.
* **Before 1970: about 1.5 units a year** (46 units, 1939-1969), so a pre-1970 listing has few portfolio peers
  on age.
* **2022: 80 units**, about four times a normal year. Real recent stock, or a default year? Section 3 checks.
* So "homogeneous on age" should mean roughly: *any year from 1970 to 2021 is typical, earlier years are
  increasingly unusual*. The scoring method has to respect that shape: a flat middle, a thin tail and a spike.

**Kernel density smoothing.** A kernel density estimate (KDE) places a small bell curve on every unit's build
year and adds them up, giving a smooth curve of how common each year is. The width of each bell (the
*bandwidth*) sets how much it smooths; `scipy` picks it from the data (Scott's rule).

In [4]:
import plotly.graph_objects as go
from scipy.stats import gaussian_kde

kde = gaussian_kde(years)
bandwidth = kde.factor * years.std()
grid = np.linspace(years.min() - 10, years.max() + 10, 400)

fig = go.Figure()
fig.add_histogram(x=years, xbins={"size": 1}, histnorm="probability density", name="build years",
                  marker_color="lightsteelblue")
fig.add_scatter(x=grid, y=kde(grid), mode="lines", name=f"KDE (bandwidth {bandwidth:.1f} years)",
                line={"color": "navy", "width": 2})
for year in marketed["build_year"].dropna():
    fig.add_vline(x=year, line_dash="dash", line_color="firebrick")
fig.update_layout(title="Portfolio build years with kernel density smoothing (dashed lines: listings)",
                  xaxis_title="Build year", yaxis_title="Share of units per year", height=420)
fig.show()
print(f"Bandwidth: {bandwidth:.1f} years")

Bandwidth: 4.0 years


* With a **bandwidth of about 4 years**, the curve follows the overall shape: the flat 1970-2021 middle, at
  roughly 1.5-2% of units per year, and the thin pre-1970 tail, falling to about 0.5% a year around 1966 (M5).
* It irons out year-to-year noise, which is what we want: a listing built in 1977 shouldn't score differently
  from one built in 1978 just because one year happened to have a few more units.
* Two problems at the **right-hand edge**:
  * the **2022 spike is smoothed away**: 6.5% of units are recorded as 2022, but the curve gives it about 1.3%,
    spreading the spike over the years either side, including years after 2022;
  * the curve **spills past 2022** into years with no buildings (still about 0.9% a year at 2024), and dips
    at 2022 itself, because there's no data beyond the edge to balance it.
* Both matter only for listings built near 2022 (none of ours), but a scoring method should handle the edge
  properly. Section 5 looks at fixing it.

**Frequency per year, scaled so the maximum is 1.** A simpler alternative that has no edge problem: count the
portfolio units built in each year, and divide by the count in the most frequent year. A listing scores the value for
its build year: 1 means "built in the portfolio's most common year".

In [5]:
per_year = years.astype(int).value_counts().reindex(range(int(years.min()), int(years.max()) + 1), fill_value=0)
year_score = per_year / per_year.max()

fig = px.bar(x=year_score.index, y=year_score.values, height=380,
             title="Score per build year: units that year / units in the most frequent year (dashed lines: listings)",
             labels={"x": "Build year", "y": "Score"})
for year in marketed["build_year"].dropna():
    fig.add_vline(x=year, line_dash="dash", line_color="firebrick")
fig.show()

print(f"Most frequent year: {per_year.idxmax()} ({per_year.max()} units). "
      f"1970-2021 scores: median {year_score.loc[1970:2021].median():.2f}, "
      f"range {year_score.loc[1970:2021].min():.2f}-{year_score.loc[1970:2021].max():.2f}")
listed = marketed[["build_year"]].dropna().astype(int)
listed["units that year"] = listed["build_year"].map(per_year)
listed["score"] = listed["build_year"].map(year_score).round(2)
listed

Most frequent year: 2022 (80 units). 1970-2021 scores: median 0.28, range 0.12-0.41


,build_year,units that year,score
listing_id,,,
M1,1977,30,0.38
M3,1976,24,0.30
M4,2000,25,0.31
M5,1966,8,0.10


* **No edge problem**: nothing spills past 2022, and 2022 keeps its full count.
* But **2022 sets the scale for every other year.** With 80 units it scores 1, so the whole flat 1970-2021
  middle is squeezed into 0.12-0.41 (a typical year scores about 0.28). A 1990s listing would read as "not very
  homogeneous" although Indurent owns plenty of 1990s stock, and if 2022 turns out to be a default year, every
  score is distorted by it.
* **Year-to-year noise shows through**: 1986, 1987 and 1988 score 0.28, 0.41 and 0.28, so a 1987 listing would
  look 50% more homogeneous than a 1986 one by chance. The KDE smoothed that away.
* The listings: M5 (1966) 0.10, M3 (1976) 0.30, M1 (1977) 0.38, M4 (2000) 0.31. M1 beats M3 only because
  1977 happens to have six more units than 1976.

**Frequency per 5-year band, scaled so the maximum is 1.** The same idea with wider bins: counting over five
years averages out the noise, and a one-year spike lifts its own band instead of setting the scale for all of
them.

In [6]:
band_start = years.astype(int) // 5 * 5
per_band = band_start.value_counts().sort_index()
band_score = per_band / per_band.max()
band_label = {start: f"{start}-{str(min(start + 4, int(years.max())))[2:]}" for start in per_band.index}

fig = px.bar(x=[band_label[s] for s in per_band.index], y=band_score.values, height=380,
             title="Score per 5-year band: units in the band / units in the most frequent band",
             labels={"x": "Build year band", "y": "Score"})
fig.update_xaxes(type="category")  # otherwise labels like "1975-79" are read as dates
fig.show()

listed["band"] = (listed["build_year"] // 5 * 5).map(band_label)
listed["band score"] = (listed["build_year"] // 5 * 5).map(band_score).round(2)
listed[["build_year", "band", "band score"]]

,build_year,band,band score
listing_id,,,
M1,1977,1975-79,0.86
M3,1976,1975-79,0.86
M4,2000,2000-04,0.86
M5,1966,1965-69,0.20


In [7]:
# Is the jump between the 1965-69 and 1970-74 bands real? Look at the single years either side.
per_year.loc[1965:1974].to_frame("units").T

build_year,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974
units,4,8,3,5,5,20,17,18,16,22


* **1970-2022 scores 0.6-1.0**; the most frequent band is 1995-99 (127 units). The 1960s score 0.05-0.20, and
  the bands before 1960 0.08 or less.
* **2022 no longer dominates**: it lifts the 2020-22 band to 0.87, among the most typical, but not above the
  rest. The last band covers only three years (there's nothing after 2022), so its count includes the spike.
* The listings: **M1 and M3 now score the same (0.86)**, as they should for buildings a year apart; **M4 also
  0.86**; **M5 0.20**. M2 has no build year, so no score.
* **The jump between 1965-69 (0.20) and 1970-74 (0.73) is in the data, not made by the bands.** The single-year
  counts jump at the same place: 1965-1969 each have a handful of units, 1970 onwards around 20 a year. So
  Indurent's stock really does start in earnest in 1970, and a 1969 building really is rarer in the portfolio
  than a 1970 one.

## 3. Decisions for `similarity.py`

We are going to use 5 year bands, rescaled so that the max of the bands has a score of 1

| Topic | Decision | Why |
|---|---|---|
| Age score | Units in the listing's 5-year build-year band, divided by units in the most frequent band (0-1) | Frequency has no edge problem at 2022; 5-year bands average out year-to-year noise and stop the 2022 spike setting the scale |
| Bands | Fixed 5-year bands (1935-39, ..., 2020-22) | The jump at 1970 is in the data, not made by the bands |
| Doubtful build years | Left out of the counts (none are flagged at present) | Doubtful values are never scored. Granite Mews 59's 1939-1953 build years were suspected but are plausible, so they count |
| No build year | Age score "not available" (M2), never 0 | A missing value says nothing about age |
| Not used | Kernel density smoothing; single-year frequency | KDE spills past 2022 and smooths away the spike; single years are noisy and let 2022 set the scale |
